# Notebook 03b: SRS Scores (MARS)

**Purpose:** Compute Session Reliability Score (SRS) per session using adapted formula for MARS.

**MARS SRS formula** (adapted — no action_types, use n_unique_items instead):
```python
# Compute CAPs from training sessions only — no leakage
CAP_INTENSITY   = np.percentile(train_sessions['n_events'], 95)
CAP_EXTENT      = np.percentile(train_sessions['duration_sec'], 95)
CAP_COMPOSITION = np.percentile(train_sessions['n_unique_items'], 95)

def compute_srs(n_events, duration_sec, n_unique_items):
    intensity   = min(n_events / CAP_INTENSITY, 1.0)
    extent      = min(duration_sec / CAP_EXTENT, 1.0)
    composition = min(n_unique_items / CAP_COMPOSITION, 1.0)
    return round((intensity + extent + composition) / 3.0, 6)
```
Note: CAP_COMPOSITION replaces N_POSSIBLE from XuetangX. Formula structure is identical.

**Inputs:**
- `data/processed/mars/sessions/sessions.parquet`
- `data/processed/mars/pairs/pairs.parquet`
- `data/processed/mars/user_splits/users_train.json` (from NB04, for CAP computation — no leakage)

**Outputs:**
- `data/processed/mars/pairs_srs/pairs.parquet` (pairs + srs column)
- `data/processed/mars/pairs_srs/session_srs.parquet`

In [1]:
# [CELL 03b-00] Bootstrap

import json
import time
import uuid
import hashlib
import math
from pathlib import Path
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd

def find_repo_root(start):
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p / 'PROJECT_STATE.md').exists(): return p
    raise RuntimeError('PROJECT_STATE.md not found')

REPO_ROOT = find_repo_root(Path.cwd())
PATHS = {
    'META_REGISTRY':  REPO_ROOT / 'meta.json',
    'DATA_PROCESSED': REPO_ROOT / 'data' / 'processed',
    'REPORTS':        REPO_ROOT / 'reports',
}

def cell_start(cid, title, **kw):
    t = time.time()
    print(f'\n[{cid}] {title}')
    print(f'[{cid}] start={datetime.now().isoformat(timespec="seconds")}')
    for k, v in kw.items(): print(f'[{cid}] {k}={v}')
    return t

def cell_end(cid, t0, **kw):
    for k, v in kw.items(): print(f'[{cid}] {k}={v}')
    print(f'[{cid}] elapsed={time.time()-t0:.2f}s  done')

def write_json_atomic(path, obj, indent=2):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f'.tmp_{uuid.uuid4().hex}')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding='utf-8')
    tmp.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(1 << 20): h.update(chunk)
    return h.hexdigest()

print(f'[CELL 03b-00] REPO_ROOT={REPO_ROOT}  done')

[CELL 03b-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta  done


In [2]:
# [CELL 03b-01] Seed + run config + paths

import random

GLOBAL_SEED = 20260107
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

NOTEBOOK_NAME = '03b_srs_scores_mars'
DATASET       = 'mars'

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ID  = uuid.uuid4().hex

OUT_DIR       = PATHS['REPORTS'] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / 'report.json'
CONFIG_PATH   = OUT_DIR / 'config.json'
MANIFEST_PATH = OUT_DIR / 'manifest.json'

# Input paths
SESSIONS_PARQUET = PATHS['DATA_PROCESSED'] / 'mars' / 'sessions' / 'sessions.parquet'
PAIRS_PARQUET    = PATHS['DATA_PROCESSED'] / 'mars' / 'pairs' / 'pairs.parquet'
USERS_TRAIN_JSON = PATHS['DATA_PROCESSED'] / 'mars' / 'user_splits' / 'users_train.json'

# Output paths
OUT_SRS_DIR   = PATHS['DATA_PROCESSED'] / 'mars' / 'pairs_srs'
OUT_SRS_DIR.mkdir(parents=True, exist_ok=True)
OUT_PAIRS_SRS   = OUT_SRS_DIR / 'pairs.parquet'
OUT_SESSION_SRS = OUT_SRS_DIR / 'session_srs.parquet'

CFG = {
    'notebook': NOTEBOOK_NAME, 'dataset': DATASET, 'seed': GLOBAL_SEED,
    'srs': {
        'cap_percentile': 95,
        'components': ['intensity', 'extent', 'composition'],
        'composition_field': 'n_unique_items',  # MARS: no action_types, use unique items
        'note': 'CAPs computed from training sessions only (no leakage)',
    },
}
write_json_atomic(CONFIG_PATH, CFG)

for path, obj in [
    (REPORT_PATH, {'run_id': RUN_ID, 'notebook': NOTEBOOK_NAME, 'run_tag': RUN_TAG,
                   'created_at': datetime.now().isoformat(timespec='seconds'),
                   'metrics': {}, 'key_findings': [], 'sanity_samples': {},
                   'data_fingerprints': {}, 'notes': []}),
    (MANIFEST_PATH, {'run_id': RUN_ID, 'notebook': NOTEBOOK_NAME, 'run_tag': RUN_TAG, 'artifacts': []}),
]:
    write_json_atomic(path, obj)

META = PATHS['META_REGISTRY']
if not META.exists(): write_json_atomic(META, {'schema_version': 1, 'runs': []})
m = read_json(META)
m['runs'].append({'run_id': RUN_ID, 'notebook': NOTEBOOK_NAME, 'run_tag': RUN_TAG, 'out_dir': str(OUT_DIR)})
write_json_atomic(META, m)

print(f'[CELL 03b-01] RUN_TAG={RUN_TAG}')
print(f'[CELL 03b-01] SESSIONS_PARQUET={SESSIONS_PARQUET}')
print(f'[CELL 03b-01] PAIRS_PARQUET={PAIRS_PARQUET}')
print(f'[CELL 03b-01] USERS_TRAIN_JSON={USERS_TRAIN_JSON}')
print('[CELL 03b-01] done')

[CELL 03b-01] RUN_TAG=20260409_190447
[CELL 03b-01] SESSIONS_PARQUET=/home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/mars/sessions/sessions.parquet
[CELL 03b-01] PAIRS_PARQUET=/home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/mars/pairs/pairs.parquet
[CELL 03b-01] USERS_TRAIN_JSON=/home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/mars/user_splits/users_train.json
[CELL 03b-01] done


In [3]:
# [CELL 03b-02] Load sessions + pairs + train user list

t0 = cell_start('CELL 03b-02', 'Load sessions + pairs + train user list')

for p in [SESSIONS_PARQUET, PAIRS_PARQUET]:
    if not p.exists():
        raise RuntimeError(f'Missing {p}. Run earlier notebooks first.')

sessions = pd.read_parquet(SESSIONS_PARQUET)
pairs_df  = pd.read_parquet(PAIRS_PARQUET)

print(f'[CELL 03b-02] sessions shape: {sessions.shape}')
print(f'[CELL 03b-02] sessions columns: {list(sessions.columns)}')
print(f'[CELL 03b-02] pairs shape:    {pairs_df.shape}')

# Verify required session columns
required_cols = ['session_id', 'user_id', 'n_events', 'duration_sec', 'n_unique_items']
missing_cols = [c for c in required_cols if c not in sessions.columns]
if missing_cols:
    raise RuntimeError(f'Missing columns in sessions: {missing_cols}. Re-run 02_sessionize_mars.ipynb.')

# Load train users for CAP computation (no leakage)
if USERS_TRAIN_JSON.exists():
    train_users = set(read_json(USERS_TRAIN_JSON))
    train_sessions = sessions[sessions['user_id'].isin(train_users)]
    print(f'[CELL 03b-02] Train users loaded: {len(train_users):,}')
    print(f'[CELL 03b-02] Train sessions (for CAP): {len(train_sessions):,}')
else:
    # NB04 not yet run — use all sessions (acceptable before split)
    train_sessions = sessions
    print('[CELL 03b-02] WARNING: users_train.json not found. Using ALL sessions for CAP computation.')
    print('[CELL 03b-02] WARNING: Re-run this notebook after 04_user_split_mars.ipynb for proper no-leakage CAPs.')

print(f'\n[CELL 03b-02] Sessions used for CAP: {len(train_sessions):,}')
cell_end('CELL 03b-02', t0)


[CELL 03b-02] Load sessions + pairs + train user list
[CELL 03b-02] start=2026-04-09T19:04:48
[CELL 03b-02] sessions shape: (561, 5)
[CELL 03b-02] sessions columns: ['session_id', 'user_id', 'n_events', 'duration_sec', 'n_unique_items']
[CELL 03b-02] pairs shape:    (2333, 7)
[CELL 03b-02] Train users loaded: 264
[CELL 03b-02] Train sessions (for CAP): 363

[CELL 03b-02] Sessions used for CAP: 363
[CELL 03b-02] elapsed=0.04s  done


In [4]:
# [CELL 03b-03] Compute CAPs from training sessions (no leakage)

t0 = cell_start('CELL 03b-03', 'Compute CAPs from training sessions')

CAP_INTENSITY   = float(np.percentile(train_sessions['n_events'], 95))
CAP_EXTENT      = float(np.percentile(train_sessions['duration_sec'], 95))
CAP_COMPOSITION = float(np.percentile(train_sessions['n_unique_items'], 95))

print(f'[CELL 03b-03] CAP_INTENSITY   (95th pct n_events):      {CAP_INTENSITY:.1f}')
print(f'[CELL 03b-03] CAP_EXTENT      (95th pct duration_sec):  {CAP_EXTENT:.1f}s ({CAP_EXTENT/60:.1f}min)')
print(f'[CELL 03b-03] CAP_COMPOSITION (95th pct n_unique_items): {CAP_COMPOSITION:.1f}')

# Sanity checks
assert CAP_INTENSITY > 0,   'CAP_INTENSITY must be > 0'
assert CAP_EXTENT > 0,      'CAP_EXTENT must be > 0'
assert CAP_COMPOSITION > 0, 'CAP_COMPOSITION must be > 0'

# Distribution stats for reference
print(f'\n[CELL 03b-03] n_events distribution (train):')
print(f'  min={train_sessions["n_events"].min()}  p50={train_sessions["n_events"].quantile(.5):.0f}  p90={train_sessions["n_events"].quantile(.9):.0f}  max={train_sessions["n_events"].max()}')
print(f'[CELL 03b-03] duration_sec distribution (train):')
print(f'  min={train_sessions["duration_sec"].min():.0f}  p50={train_sessions["duration_sec"].quantile(.5):.0f}  p90={train_sessions["duration_sec"].quantile(.9):.0f}  max={train_sessions["duration_sec"].max():.0f}')
print(f'[CELL 03b-03] n_unique_items distribution (train):')
print(f'  min={train_sessions["n_unique_items"].min()}  p50={train_sessions["n_unique_items"].quantile(.5):.0f}  p90={train_sessions["n_unique_items"].quantile(.9):.0f}  max={train_sessions["n_unique_items"].max()}')

cell_end('CELL 03b-03', t0, cap_intensity=CAP_INTENSITY, cap_extent=CAP_EXTENT, cap_composition=CAP_COMPOSITION)


[CELL 03b-03] Compute CAPs from training sessions
[CELL 03b-03] start=2026-04-09T19:04:48
[CELL 03b-03] CAP_INTENSITY   (95th pct n_events):      16.9
[CELL 03b-03] CAP_EXTENT      (95th pct duration_sec):  3757.6s (62.6min)
[CELL 03b-03] CAP_COMPOSITION (95th pct n_unique_items): 16.9

[CELL 03b-03] n_events distribution (train):
  min=2  p50=3  p90=10  max=47
[CELL 03b-03] duration_sec distribution (train):
  min=10  p50=463  p90=2318  max=13098
[CELL 03b-03] n_unique_items distribution (train):
  min=2  p50=3  p90=10  max=47
[CELL 03b-03] cap_intensity=16.899999999999977
[CELL 03b-03] cap_extent=3757.5999999999967
[CELL 03b-03] cap_composition=16.899999999999977
[CELL 03b-03] elapsed=0.00s  done


In [5]:
# [CELL 03b-04] Compute SRS per session (MARS adapted formula)

t0 = cell_start('CELL 03b-04', 'Compute SRS scores')

def compute_srs(n_events, duration_sec, n_unique_items):
    """
    MARS SRS formula (adapted — no action_types, use n_unique_items instead):
      intensity   = min(n_events / CAP_INTENSITY, 1.0)
      extent      = min(duration_sec / CAP_EXTENT, 1.0)
      composition = min(n_unique_items / CAP_COMPOSITION, 1.0)
      SRS         = (intensity + extent + composition) / 3
    CAPs computed from training sessions only — no leakage.
    """
    intensity   = min(n_events / CAP_INTENSITY, 1.0)
    extent      = min(duration_sec / CAP_EXTENT, 1.0)
    composition = min(n_unique_items / CAP_COMPOSITION, 1.0)
    return round((intensity + extent + composition) / 3.0, 6)

sessions['srs'] = sessions.apply(
    lambda r: compute_srs(r['n_events'], r['duration_sec'], r['n_unique_items']),
    axis=1
)

# Individual components
sessions['srs_intensity']   = (sessions['n_events']       / CAP_INTENSITY).clip(upper=1.0)
sessions['srs_extent']      = (sessions['duration_sec']   / CAP_EXTENT).clip(upper=1.0)
sessions['srs_composition'] = (sessions['n_unique_items'] / CAP_COMPOSITION).clip(upper=1.0)

srs_vals = sessions['srs']
print(f'[CELL 03b-04] SRS distribution:')
print(f'  mean={srs_vals.mean():.4f}  std={srs_vals.std():.4f}')
print(f'  min={srs_vals.min():.4f}  p25={srs_vals.quantile(.25):.4f}  p50={srs_vals.quantile(.5):.4f}')
print(f'  p75={srs_vals.quantile(.75):.4f}  p90={srs_vals.quantile(.9):.4f}  max={srs_vals.max():.4f}')

low    = float((srs_vals < 0.33).mean())
medium = float(((srs_vals >= 0.33) & (srs_vals < 0.66)).mean())
high   = float((srs_vals >= 0.66).mean())
print(f'\n[CELL 03b-04] Tier breakdown:')
print(f'  low (<0.33):    {low:.1%}  ({(srs_vals < 0.33).sum():,} sessions)')
print(f'  medium (0.33-0.66): {medium:.1%}  ({((srs_vals >= 0.33) & (srs_vals < 0.66)).sum():,} sessions)')
print(f'  high (>=0.66):  {high:.1%}  ({(srs_vals >= 0.66).sum():,} sessions)')

print(f'\n[CELL 03b-04] Component means:')
print(f'  srs_intensity:   {sessions["srs_intensity"].mean():.4f}')
print(f'  srs_extent:      {sessions["srs_extent"].mean():.4f}')
print(f'  srs_composition: {sessions["srs_composition"].mean():.4f}')

cell_end('CELL 03b-04', t0, srs_mean=float(srs_vals.mean()), srs_p50=float(srs_vals.quantile(.5)))


[CELL 03b-04] Compute SRS scores
[CELL 03b-04] start=2026-04-09T19:04:48
[CELL 03b-04] SRS distribution:
  mean=0.2665  std=0.2415
  min=0.0791  p25=0.0995  p50=0.1627
  p75=0.3130  p90=0.6280  max=1.0000

[CELL 03b-04] Tier breakdown:
  low (<0.33):    76.5%  (429 sessions)
  medium (0.33-0.66): 14.3%  (80 sessions)
  high (>=0.66):  9.3%  (52 sessions)

[CELL 03b-04] Component means:
  srs_intensity:   0.2764
  srs_extent:      0.2468
  srs_composition: 0.2764
[CELL 03b-04] srs_mean=0.2665287147950089
[CELL 03b-04] srs_p50=0.162698
[CELL 03b-04] elapsed=0.02s  done


In [6]:
# [CELL 03b-05] Join SRS to pairs + validate

t0 = cell_start('CELL 03b-05', 'Join SRS to pairs')

srs_map = sessions.set_index('session_id')[['srs', 'srs_intensity', 'srs_extent', 'srs_composition']]
pairs_srs = pairs_df.merge(srs_map, on='session_id', how='left')

n_missing = int(pairs_srs['srs'].isna().sum())
if n_missing > 0:
    print(f'[CELL 03b-05] WARNING: {n_missing} pairs missing SRS (filling with 0.0)')
    pairs_srs['srs']             = pairs_srs['srs'].fillna(0.0)
    pairs_srs['srs_intensity']   = pairs_srs['srs_intensity'].fillna(0.0)
    pairs_srs['srs_extent']      = pairs_srs['srs_extent'].fillna(0.0)
    pairs_srs['srs_composition'] = pairs_srs['srs_composition'].fillna(0.0)
else:
    print(f'[CELL 03b-05] All pairs have SRS — no missing values')

print(f'[CELL 03b-05] pairs_srs shape: {pairs_srs.shape}')
print(f'[CELL 03b-05] SRS in pairs: mean={pairs_srs["srs"].mean():.4f}  p50={pairs_srs["srs"].quantile(.5):.4f}')

# Validate all SRS in [0, 1]
assert (pairs_srs['srs'] >= 0.0).all(), 'SRS below 0 found'
assert (pairs_srs['srs'] <= 1.0).all(), 'SRS above 1 found'
print('[CELL 03b-05] Assertion passed: all SRS scores in [0, 1]')

# Verify all original pair columns preserved
for col in pairs_df.columns:
    assert col in pairs_srs.columns, f'Column {col} missing from pairs_srs'
print(f'[CELL 03b-05] All original pair columns preserved: {list(pairs_df.columns)}')
print(f'[CELL 03b-05] New columns added: srs, srs_intensity, srs_extent, srs_composition')

cell_end('CELL 03b-05', t0, n_pairs=int(len(pairs_srs)), n_missing=n_missing)


[CELL 03b-05] Join SRS to pairs
[CELL 03b-05] start=2026-04-09T19:04:48
[CELL 03b-05] All pairs have SRS — no missing values
[CELL 03b-05] pairs_srs shape: (2333, 11)
[CELL 03b-05] SRS in pairs: mean=0.5817  p50=0.5610
[CELL 03b-05] Assertion passed: all SRS scores in [0, 1]
[CELL 03b-05] All original pair columns preserved: ['pair_id', 'user_id', 'session_id', 'prefix', 'label', 'label_ts_epoch', 'prefix_len']
[CELL 03b-05] New columns added: srs, srs_intensity, srs_extent, srs_composition
[CELL 03b-05] n_pairs=2333
[CELL 03b-05] n_missing=0
[CELL 03b-05] elapsed=0.00s  done


In [7]:
# [CELL 03b-06] Save outputs + sha256 fingerprints + report

t0 = cell_start('CELL 03b-06', 'Save pairs_srs + session_srs + report')

# Save pairs_srs/pairs.parquet
pairs_srs.to_parquet(OUT_PAIRS_SRS, index=False, compression='zstd')
pairs_srs_bytes = int(OUT_PAIRS_SRS.stat().st_size)
pairs_srs_sha   = sha256_file(OUT_PAIRS_SRS)

print(f'[CELL 03b-06] Saved {OUT_PAIRS_SRS} ({pairs_srs_bytes / (1 << 20):.2f} MB)')
print(f'[CELL 03b-06] SHA256 (pairs_srs): {pairs_srs_sha}')

# Save pairs_srs/session_srs.parquet
session_srs_cols = ['session_id', 'user_id', 'n_events', 'duration_sec', 'n_unique_items',
                    'srs', 'srs_intensity', 'srs_extent', 'srs_composition']
session_srs_df = sessions[session_srs_cols].copy()
session_srs_df.to_parquet(OUT_SESSION_SRS, index=False, compression='zstd')
sess_srs_bytes = int(OUT_SESSION_SRS.stat().st_size)
sess_srs_sha   = sha256_file(OUT_SESSION_SRS)

print(f'[CELL 03b-06] Saved {OUT_SESSION_SRS} ({sess_srs_bytes / (1 << 20):.2f} MB)')
print(f'[CELL 03b-06] SHA256 (session_srs): {sess_srs_sha}')

# Print summary
print(f'\n[CELL 03b-06] Summary:')
print(f'  n_sessions:          {len(session_srs_df):,}')
print(f'  n_pairs_with_srs:    {len(pairs_srs):,}')
print(f'  CAP_INTENSITY:       {CAP_INTENSITY:.1f}')
print(f'  CAP_EXTENT:          {CAP_EXTENT:.1f}s')
print(f'  CAP_COMPOSITION:     {CAP_COMPOSITION:.1f}')
print(f'  srs mean:            {pairs_srs["srs"].mean():.4f}')
print(f'  srs p50:             {pairs_srs["srs"].quantile(.5):.4f}')

# Update report
r = read_json(REPORT_PATH)
r['metrics'] = {
    'n_pairs':          int(len(pairs_srs)),
    'n_sessions':       int(len(sessions)),
    'cap_intensity':    CAP_INTENSITY,
    'cap_extent':       CAP_EXTENT,
    'cap_composition':  CAP_COMPOSITION,
    'srs_mean':         float(srs_vals.mean()),
    'srs_std':          float(srs_vals.std()),
    'srs_p50':          float(srs_vals.quantile(.5)),
    'srs_p75':          float(srs_vals.quantile(.75)),
    'tier_low':         low,
    'tier_medium':      medium,
    'tier_high':        high,
    'n_missing_srs':    n_missing,
}
r['data_fingerprints']['pairs_srs'] = {
    'path': str(OUT_PAIRS_SRS), 'bytes': pairs_srs_bytes, 'sha256': pairs_srs_sha
}
r['data_fingerprints']['session_srs'] = {
    'path': str(OUT_SESSION_SRS), 'bytes': sess_srs_bytes, 'sha256': sess_srs_sha
}
r['sanity_samples']['session_srs_head3'] = session_srs_df.head(3).to_dict(orient='records')
# Convert to JSON-safe types (prefix/label may be numpy arrays)
def to_json_safe(records):
    safe = []
    for rec in records:
        safe.append({k: (v.tolist() if hasattr(v, 'tolist') else v) for k, v in rec.items()})
    return safe
r['sanity_samples']['session_srs_head3'] = to_json_safe(session_srs_df.head(3).to_dict(orient='records'))
r['sanity_samples']['pairs_srs_head3']   = to_json_safe(pairs_srs.head(3).to_dict(orient='records'))
r['notes'].append(
    'MARS SRS: composition = min(n_unique_items / CAP_COMPOSITION, 1.0). '
    'No action_types in MARS — adapted formula uses item diversity instead. '
    f'CAPs from {"training sessions" if USERS_TRAIN_JSON.exists() else "ALL sessions (re-run after NB04)"}. '
)
write_json_atomic(REPORT_PATH, r)

# Update manifest
manifest = read_json(MANIFEST_PATH)
for path in [OUT_PAIRS_SRS, OUT_SESSION_SRS]:
    try: sha = sha256_file(path)
    except Exception as e: sha = f'ERROR: {e}'
    manifest['artifacts'].append({'path': str(path), 'bytes': int(path.stat().st_size), 'sha256': sha})
write_json_atomic(MANIFEST_PATH, manifest)

print(f'\n[CELL 03b-06] Updated: {REPORT_PATH}')
cell_end('CELL 03b-06', t0)


[CELL 03b-06] Save pairs_srs + session_srs + report
[CELL 03b-06] start=2026-04-09T19:04:48
[CELL 03b-06] Saved /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/mars/pairs_srs/pairs.parquet (0.05 MB)
[CELL 03b-06] SHA256 (pairs_srs): 18c32bc2a5c3019b1afd11aa6f89d687b1d9a676fe01683056b2baac08e5d39a
[CELL 03b-06] Saved /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/mars/pairs_srs/session_srs.parquet (0.02 MB)
[CELL 03b-06] SHA256 (session_srs): 57e8aa5fe00b7cad9cdd88ce5a0d1fd551c97a6578ad5bf7d9bf3e5daa3ef3d0

[CELL 03b-06] Summary:
  n_sessions:          561
  n_pairs_with_srs:    2,333
  CAP_INTENSITY:       16.9
  CAP_EXTENT:          3757.6s
  CAP_COMPOSITION:     16.9
  srs mean:            0.5817
  srs p50:             0.5610

[CELL 03b-06] Updated: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/03b_srs_scores_mars/20260409_190447/report.json
[CELL 03b-06] elapsed=0.02s  done


## Notebook 03b Complete

**Outputs:**
- `data/processed/mars/pairs_srs/pairs.parquet` — pairs with SRS column (intensity, extent, composition components)
- `data/processed/mars/pairs_srs/session_srs.parquet` — session-level SRS scores

**MARS SRS formula:**
- `intensity = min(n_events / CAP_INTENSITY, 1.0)`
- `extent = min(duration_sec / CAP_EXTENT, 1.0)`
- `composition = min(n_unique_items / CAP_COMPOSITION, 1.0)` ← adapted (no action_types in MARS)
- `SRS = (intensity + extent + composition) / 3`

**Important:** If `users_train.json` was missing when this ran, re-run after `04_user_split_mars.ipynb` to compute CAPs from training sessions only (no leakage).

**Next:** `04_user_split_mars.ipynb` — split users 70/15/15, then re-run this notebook.